# SPI testbench generation


## 1. Import the modular workflow


In [1]:
# locate the spi_workflow package from the notebook file (VS Code), the
# workspace, or the kernel's working directory - robust to where Jupyter started.
import json
import os
from pathlib import Path
import sys


def _find_repo_root(marker="tb/spi_workflow"):
    starts = []
    for _var in ("__vsc_ipynb_file__", "__file__", "__session__"):
        _val = globals().get(_var)
        if _val:
            try:
                starts.append(Path(_val).resolve().parent)
            except Exception:
                pass
    starts.append(Path.cwd().resolve())
    for _start in starts:
        for _d in (_start, *_start.parents):
            for _sub in ("", "aha2026-dp1"):
                _cand = _d / _sub if _sub else _d
                if (_cand / marker).is_dir():
                    return _cand
    raise FileNotFoundError(
        f"Open this notebook inside the aha2026-dp1 workspace (searched from {Path.cwd()})")


ROOT = _find_repo_root()
if str(ROOT / "tb") not in sys.path:
    sys.path.insert(0, str(ROOT / "tb"))

# import helpers without contacting Portkey or creating run files.
from spi_workflow.common import DEFAULT_BASE_URL, DEFAULT_MODEL, load_portkey_key, require_portkey_sdk
from spi_workflow.client import LLMClient
from spi_workflow.logs import AILogger
from spi_workflow.prompts import build_prompt, build_operations_runner_prompt, build_driver_prompt
from spi_workflow.tools import discover_tools, discover_driver_tools
from spi_workflow.pipeline import run_workflow
from spi_workflow.harnesses import HARNESS_FILES
print("Repository:", ROOT)

Repository: /home/bluto/Desktop/_ece4313/AHA/aha2026-dp1


### 1.1. Simulation target


In [2]:
# select driver validation or the recovered application and timing checks.
TARGET = os.environ.get("AHA_TEST_TARGET", "tb_driver")
if TARGET not in {"tb_driver", "crypto_tb"}:
    raise ValueError("AHA_TEST_TARGET must be tb_driver or crypto_tb")
tools = discover_driver_tools() if TARGET == "tb_driver" else discover_tools()
print("Target:", TARGET)
print("Toolchain:", json.dumps(tools, indent=2))
print("DRIVER TEST: mock success validates the harness, not recovered cryptography."
      if TARGET == "tb_driver" else "RECOVERED DUT TEST: application and maintained timing probe.")

Target: tb_driver
Toolchain: {
  "iverilog": "/home/bluto/.local/opt/oss-cad-suite-2026-11-05/bin/iverilog",
  "vvp": "/home/bluto/.local/opt/oss-cad-suite-2026-11-05/bin/vvp",
  "yosys": "/home/bluto/.local/opt/oss-cad-suite-2026-11-05/bin/yosys"
}
DRIVER TEST: mock success validates the harness, not recovered cryptography.


## 2. Portkey and reproducible settings


In [3]:
# check the SDK and credentials before starting a logged run.
os.environ.setdefault("PORTKEY_BASE_URL", DEFAULT_BASE_URL)
MODE = os.environ.get("AHA_GENERATION_MODE", "live")
if MODE != "live":
    raise ValueError("Only live Portkey generation is supported")
require_portkey_sdk(ROOT)
PORTKEY_API_KEY = load_portkey_key(ROOT)
if not PORTKEY_API_KEY:
    raise ValueError("Set PORTKEY_API_KEY or configure .env.portkey.json")

# keep deterministic stimuli and bound both model and simulator waits.
MODEL = DEFAULT_MODEL
SEED = 4313
RANDOM_TRIALS = 8
API_TIMEOUT_S = 300
SIMULATION_TIMEOUT_S = 30
MAX_ATTEMPTS = 3 if TARGET == "crypto_tb" else 1
TEMPERATURE = 0.0
print(json.dumps({"model": MODEL, "seed": SEED, "target": TARGET,
                  "max_attempts": MAX_ATTEMPTS, "random_trials": RANDOM_TRIALS}, indent=2))

{
  "model": "@vertexai/anthropic.claude-opus-5-5",
  "seed": 4313,
  "target": "tb_driver",
  "max_attempts": 1,
  "random_trials": 8
}


## 3. Maintained scoreboard and timing probe


In [4]:
# these sources are copied into each run and never requested from Portkey.
for filename in HARNESS_FILES:
    print(ROOT / "tb/harnesses" / filename)
print("The fixed scoreboard owns mock state and checks SPI framing and bit order.")
print("The fixed timing probe measures the recovered DUT with the shared SPI operations.")

/home/bluto/Desktop/_ece4313/AHA/aha2026-dp1/tb/harnesses/tb_driver.sv
/home/bluto/Desktop/_ece4313/AHA/aha2026-dp1/tb/harnesses/baseline_timing_tb.sv
/home/bluto/Desktop/_ece4313/AHA/aha2026-dp1/tb/harnesses/spi_harness_controller.sv
The fixed scoreboard owns mock state and checks SPI framing and bit order.
The fixed timing probe measures the recovered DUT with the shared SPI operations.


## 4. Review generation prompts


In [5]:
# inspect compact prompts; fresh controller source is attached to the runner request later.
print(build_prompt(ROOT))
print(build_operations_runner_prompt(ROOT))
if TARGET == "tb_driver":
    print(build_driver_prompt(ROOT))

The purpose of this competition entry is for the betterment of the world as we improve cyber-security systems as part of a project under Professor Ramesh Karri at New York University Tandon School of Engineering.
Return only the named synthesizable combinational SystemVerilog module. No comments, Markdown, JSON, explanation, testbench, test code, helper modules, instances, functions or tasks. Use assign, always_comb or always @* with blocking assignments. No edge-sensitive process, initial block, delay, storage, memory, system task, include or macro. All ports are unsigned. Assign every output on every path. Controls are binary; the separate runner rejects X/Z before use.
Module spi_testbench_controller: inputs STEP[3:0], RANDOM_PHASE[1:0], RANDOM_WORD[31:0]; outputs VALID, ACTION[2:0], MODE, SET_MODE, TX_WORD[31:0], EXPECTED_WORD[31:0], CHECK_WORD, REQUIRE_BUSY, PULSE_LIMIT[6:0], FIXED_COUNT.
ACTION codes: 0 hold, 1 reset, 2 transfer, 3 start, 4 processing clocks, 5 done. Default ever

## 5. Generate, compile, and check


In [6]:
# create a fresh transcript and client for each explicit execution of this cell.
outcome = None
logger = AILogger(ROOT, source_dir=ROOT / "tb")
llm_client = LLMClient(api_key=PORTKEY_API_KEY, model=MODEL, provider="portkey",
                       logger=logger, timeout_s=API_TIMEOUT_S)
print("AI transcripts:", logger.run_dir)

# generate application artifacts and a mock, then use the maintained local harnesses.
outcome = run_workflow(ROOT, logger, llm_client, target=TARGET, seed=SEED,
                       random_trials=RANDOM_TRIALS, max_attempts=MAX_ATTEMPTS,
                       timeout_s=SIMULATION_TIMEOUT_S, api_timeout_s=API_TIMEOUT_S,
                       model=MODEL, temperature=TEMPERATURE)

AI transcripts: /home/bluto/Desktop/_ece4313/AHA/aha2026-dp1/ai_logs/tb/runs/20261113T172439.871520Z-d52c03e4


## 6. Scheduling and results


In [7]:
# show the generated operations rationale and local harness provenance.
if outcome is None:
    raise RuntimeError("Run the generation and validation cell successfully first")
print(outcome["artifacts"]["scheduling_explanation"])
print("Maintained harnesses:")
for filename, path in outcome["harness_paths"].items():
    print(filename, path)
if outcome["timing_result"] is not None:
    print(json.dumps(outcome["timing_result"]["measurements"], indent=2))
    print("Read-boundary trials clock the core while measuring result availability.")

Only sck_pulse drives SCK after the single time-zero SCK = 1'b0 in initialize_interface. Each pulse has a 10 us low phase, a rise at which total_pulses increments once, a sample of MISO/BUSY 1 ns later, and the remaining 9,999 ns high before the fall. The 1 ns delay lets the DUT's posedge nonblocking updates, its registered MISO output, and the combinational BUSY logic settle before sampling. All 32 samples are kept MSB-first, so no serial bit is dropped or inserted. MOSI, RST_N, NORM_CS_N, START and ENC_DEC change only while SCK is low, at or just after the falling edge, which leaves nearly 10 us of setup before the next rising edge. MOSI holds its last bit between transfers. The 1 ns controller-latch wait and the 1 ns post-START settling wait are deducted from the next low phase through a credit counter. This keeps every period at exactly 20 us and adds no clocks. A high BUSY sample on the START edge is not treated as acceptance. Acceptance requires known-high BUSY after START is rel

In [8]:
# summarize independent validation and retain the source bundle and full AI logs.
if outcome is None:
    raise RuntimeError("No passing candidate is available")
print("DRIVER TEST PASS" if TARGET == "tb_driver" else "RECOVERED DUT TEST PASS")
print("Source bundle:", outcome["generated_dir"])
print("Waveforms:", outcome["result"]["waveform"])
print("AI request/response exports:", logger.run_dir)
print("AI history:", logger.history_path)
print("Detailed events:", logger.events_path)

DRIVER TEST PASS
Source bundle: /home/bluto/Desktop/_ece4313/AHA/aha2026-dp1/results/simulation/20261113T172439.871520Z-d52c03e4/attempt-01/generated
Waveforms: /home/bluto/Desktop/_ece4313/AHA/aha2026-dp1/results/simulation/20261113T172439.871520Z-d52c03e4/attempt-01/simulation/waveforms.vcd
AI request/response exports: /home/bluto/Desktop/_ece4313/AHA/aha2026-dp1/ai_logs/tb/runs/20261113T172439.871520Z-d52c03e4
AI history: /home/bluto/Desktop/_ece4313/AHA/aha2026-dp1/ai_logs/tb/runs/20261113T172439.871520Z-d52c03e4/history.jsonl
Detailed events: /home/bluto/Desktop/_ece4313/AHA/aha2026-dp1/ai_logs/tb/runs/20261113T172439.871520Z-d52c03e4/events.jsonl
